# 02 — Querying with SELECT

**What's in this notebook:**
- The shape of a `SELECT` statement and which clauses are optional
- Choosing what comes out — columns, expressions, function calls, and aliases
- Removing duplicates with `DISTINCT` (and what "distinct" actually compares)
- `NULL` in expressions — three-valued logic, `COALESCE`, `NULLIF`
- The logical clause order — why `WHERE` can't see a `SELECT` alias but `ORDER BY` can
- Common pitfalls to carry forward

This notebook reuses the `customers` and `orders` tables created in notebook 01. If you haven't run that one yet, run it first — the tables persist inside `learn.db`.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

## 1. The shape of a SELECT statement

Every `SELECT` has the same skeleton. Only the first two clauses are mandatory:

```
SELECT     <expressions>           -- what comes out
FROM       <source>                -- where rows come from
[WHERE     <row filter>]           -- which rows to keep
[GROUP BY  <columns>]              -- collapse rows into groups
[HAVING    <group filter>]         -- which groups to keep
[ORDER BY  <columns>]              -- sort the result
[LIMIT     <n> [OFFSET <m>]]       -- slice the result
```

Each clause has a single job. This notebook focuses on **`SELECT`** and **`FROM`**. The rest get their own notebooks — filtering and sorting in 03, aggregation and `GROUP BY` in 05, and so on.

## 2. Choosing what comes out

The `SELECT` clause is a **projection** — it picks which columns and expressions end up in the result. You can list:

- **Existing columns** — `SELECT name, email FROM customers;`
- **All columns** — `SELECT * FROM customers;` (fine for ad-hoc queries, avoid in saved code)
- **Literals** — `SELECT 1 + 1;` or `SELECT 'hello';` (no `FROM` needed)
- **Expressions** — arithmetic, string concatenation, date math
- **Function calls** — `UPPER(name)`, `LENGTH(name)`, `EXTRACT(YEAR FROM signup_date)`
- **Aliases** — give any expression a result-column name with `AS` (the `AS` is optional but reads better)

Aliases matter because the engine has to label every output column, and a raw expression like `amount * 1.08` becomes an awkward auto-generated name.

In [ ]:
%%sql
-- Pick specific columns, alias one of them.
SELECT customer_id AS id,
       name,
       email
FROM customers;

In [ ]:
%%sql
-- Expressions, function calls, and a literal — all valid in SELECT.
SELECT order_id,
       amount,
       amount * 1.08         AS amount_with_tax,
       ROUND(amount * 1.08, 2) AS amount_with_tax_rounded,
       EXTRACT(YEAR FROM order_date) AS order_year,
       'USD'                 AS currency
FROM orders;

## 3. Removing duplicates with DISTINCT

`DISTINCT` deduplicates the result set. Two things to internalize:

1. **`DISTINCT` applies to the whole row of the projection, not per column.** `SELECT DISTINCT a, b` returns distinct *pairs* of `(a, b)`, not distinct `a` and separately distinct `b`.
2. **It is not free.** The engine has to sort or hash every row to find duplicates. On large tables, ask whether you actually have duplicates, or whether you really want `GROUP BY` (notebook 05).

In [ ]:
%%sql
-- Distinct customer_ids that appear in orders.
SELECT DISTINCT customer_id FROM orders;

In [ ]:
%%sql
-- DISTINCT on a pair: distinct (customer_id, order_year) combinations.
SELECT DISTINCT customer_id, EXTRACT(YEAR FROM order_date) AS order_year
FROM orders;

## 4. NULL in expressions — three-valued logic

`NULL` means *value unknown*. It is not zero. It is not the empty string. It is the absence of a value.

That choice has one big consequence: SQL uses **three-valued logic**. Every boolean expression evaluates to **TRUE**, **FALSE**, or **UNKNOWN**, and `UNKNOWN` is what you get whenever `NULL` is involved.

- Arithmetic: `NULL + 1` → `NULL`
- String concat: `NULL || 'x'` → `NULL` (in standard SQL)
- Comparison: `NULL = NULL` → `UNKNOWN` (treated as not-true by `WHERE`)
- Negation: `NOT UNKNOWN` → `UNKNOWN`

Two functions handle NULL explicitly:

- **`COALESCE(a, b, c, ...)`** — returns the first argument that is not NULL.
- **`NULLIF(a, b)`** — returns NULL if `a = b`, otherwise returns `a`. Handy for turning sentinel values (like `''` or `-1`) into proper NULLs.

In [ ]:
%%sql
-- NULL propagates through arithmetic and comparison.
SELECT NULL + 1                AS arith,
       NULL = NULL             AS null_equals_null,
       NULL IS NULL            AS null_is_null,
       COALESCE(NULL, NULL, 'fallback') AS coalesced,
       NULLIF('', '')          AS nullif_empty;

In [ ]:
%%sql
-- Real use: substitute a friendly value for missing emails.
SELECT name,
       COALESCE(email, '(no email on file)') AS contact
FROM customers;

## 5. The logical clause order

You **write** a query in this order: `SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT`.

But the engine **executes** it in a different order. Internalize this — it explains most of SQL's surprises.

```
   1. FROM       — pick the source(s)
   2. WHERE      — filter rows out
   3. GROUP BY   — collapse remaining rows into groups
   4. HAVING     — filter whole groups out
   5. SELECT     — project columns / compute expressions / assign aliases
   6. ORDER BY   — sort the result
   7. LIMIT      — keep only the top N rows
```

Three consequences fall straight out of this order:

- **`WHERE` cannot see `SELECT` aliases** — `WHERE` runs at step 2, `SELECT` (where aliases are created) runs at step 5. The alias does not exist yet.
- **`ORDER BY` *can* see `SELECT` aliases** — `ORDER BY` runs at step 6, after `SELECT` has produced them.
- **Filter aggregates with `HAVING`, not `WHERE`** — aggregates like `SUM` and `COUNT` only exist after `GROUP BY` (step 3). A `WHERE SUM(...) > 100` is a category error and the engine will reject it.

We will exercise filtering and sorting properly in notebook 03; the point here is the mental model.

In [ ]:
%%sql
-- ORDER BY can reference a SELECT alias — SELECT has already run.
SELECT name,
       LENGTH(name) AS name_length
FROM customers
ORDER BY name_length DESC;

In [ ]:
%%sql
-- WHERE cannot reference a SELECT alias — this fails.
-- Workaround: repeat the expression, or wrap in a subquery (notebook 06).
SELECT name,
       LENGTH(name) AS name_length
FROM customers
WHERE name_length > 3;

## 6. Common pitfalls (carry forward)

- **Referencing a `SELECT` alias inside `WHERE`** — the alias does not exist yet at filter time. Either repeat the expression, or wrap the query in a subquery / CTE.
- **Expecting `DISTINCT` to deduplicate per column** — it deduplicates the *whole projected row*. If you want "distinct values of `a` regardless of `b`", project only `a`.
- **Using `= NULL` instead of `IS NULL`** — the comparison evaluates to `UNKNOWN`, which `WHERE` treats as not-true, so no rows come back. Always use `IS NULL` / `IS NOT NULL`.
- **Leaning on `SELECT *` in saved code** — when columns change, downstream code breaks silently. Name what you need; let the projection be the contract.
- **Forgetting `ORDER BY`** — the engine is free to return rows in any order. If your test relies on order, the test will be flaky.

Next up: **notebook 03 — Filtering, Sorting & Set Operations**, where `WHERE`, `ORDER BY`, `LIMIT`, and the set operators (`UNION`, `INTERSECT`, `EXCEPT`) get the deep treatment.